In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from astroquery.mast import Observations

DATA_DIR = Path("data")
FITS_DIR = DATA_DIR / "miri_i2d"  # MAST Level-3 *_i2d.fits
# Broad-band filters useful for high-z continuum / break / line work (see MIRI_plan.md).
PREFERRED_FILTERS = {
    "F560W", "F770W", "F1000W", "F1130W", "F1280W",
    "F1500W", "F1800W", "F2100W", "F2550W",
}
MATCH_RADIUS = 60 * u.arcsec  # imaging FOV scale

## 1. Get MIRI imaging in MAST

In [2]:
obs = Observations.query_criteria(
    obs_collection="JWST",
    instrument_name="MIRI/IMAGE",
)
miri = SkyCoord(obs["s_ra"], obs["s_dec"], unit="deg")
print(f"{len(obs)} MIRI/IMAGE observations in MAST")
obs[:3]

20160 MIRI/IMAGE observations in MAST


intentType,obs_collection,provenance_name,instrument_name,project,filters,wave_region,target_name,target_classification,obs_id,s_ra,s_dec,dataproduct_type,proposal_pi,calib_level,t_min,t_max,t_exptime,wavelength_region,em_min,em_max,obs_title,t_obs_release,proposal_id,proposal_type,sequence_number,s_region,jpegURL,dataURL,dataRights,mtFlag,srcDen,obsid,objID,wave_min,wave_max
str11,str4,str7,str10,str4,str7,str8,str30,str130,str42,float64,float64,str10,str33,int64,float64,float64,float64,str8,float64,float64,str160,float64,str5,str6,int64,str9974,str68,str73,str16,bool,float64,str9,str10,float64,float64
science,JWST,CALJWST,MIRI/IMAGE,JWST,F560W,Infrared,MIRIM-FP2MIMF,--,jw01465079001_02101_00001_mirimage,94.77756,-64.34610833333335,image,"Perrin, Marshall",2,59688.10633595093,59688.10701043982,58.276,Infrared,5000.0,6200.0,MIMF Measurements-2,59774.8541666,1465,COM,--,POLYGON 94.774948962 -64.341757238 94.837984827 -64.357142622 94.802784677 -64.384614781 94.739410148 -64.369531049,mast:JWST/product/jw01465079001_02101_00001_mirimage_cal.jpg,mast:JWST/product/jw01465079001_02101_00001_mirimage_cal.fits,PUBLIC,False,nan,79431376,317684473,5000.0,6200.0
science,JWST,CALJWST,MIRI/IMAGE,JWST,F560W,Infrared,MIRIM-FP2MIMF,--,jw01465079001_02101_00002_mirimage,94.77756,-64.34610833333335,image,"Perrin, Marshall",2,59688.10797372871,59688.1086482176,58.276,Infrared,5000.0,6200.0,MIMF Measurements-2,59774.8541666,1465,COM,--,POLYGON 94.775180122 -64.341636994 94.838216185 -64.357022015 94.803017033 -64.384494376 94.739642317 -64.369411009,mast:JWST/product/jw01465079001_02101_00002_mirimage_cal.jpg,mast:JWST/product/jw01465079001_02101_00002_mirimage_cal.fits,PUBLIC,False,nan,79431377,317684529,5000.0,6200.0
science,JWST,CALJWST,MIRI/IMAGE,JWST,F560W,Infrared,MIRIM-FP3MIMF,--,jw01465080001_02101_00001_mirimage,94.77756,-64.34610833333335,image,"Perrin, Marshall",2,59688.11651816157,59688.11719265046,58.276,Infrared,5000.0,6200.0,MIMF Measurements-2,59774.8541666,1465,COM,--,POLYGON 94.793929148 -64.326987061 94.85693653 -64.342368343 94.821764829 -64.36984279 94.758419083 -64.354763193,mast:JWST/product/jw01465080001_02101_00001_mirimage_cal.jpg,mast:JWST/product/jw01465080001_02101_00001_mirimage_cal.fits,PUBLIC,False,nan,79431378,317684614,5000.0,6200.0


## 2. Load DJA NIRSpec sources

In [3]:
dja = Table.read("data/dja_nirspec_z>9_grade3.csv")
dja_coords = SkyCoord(dja["ra"], dja["dec"], unit="deg")
print(f"{len(dja)} DJA sources, {len(obs)} MIRI imaging observations")

177 DJA sources, 20160 MIRI imaging observations


## 3. Match within the MIRI imager FOV

Nearest MIRI/IMAGE pointing per DJA row; keep if separation < `MATCH_RADIUS` (default 60″). MIRI imager FOV is ~74″×113″.

In [4]:
# Nearest MIRI imaging pointing per DJA row
idx, sep2d, _ = dja_coords.match_to_catalog_sky(miri)
matched = sep2d < MATCH_RADIUS

hits = dja[matched].copy()
hits["sep_arcsec"] = sep2d[matched].to(u.arcsec).value
for col in ["obsid", "proposal_id", "instrument_name", "obs_id", "s_ra", "s_dec",
            "t_exptime", "filters", "dataproduct_type"]:
    if col in obs.colnames:
        hits[f"miri_{col}"] = obs[col][idx[matched]]

hits.sort("sep_arcsec")

# Collapse duplicate DJA entries at the same sky position; keep closest MIRI match
unique = hits.copy()
unique["ra_round"] = np.round(unique["ra"].astype(float), 6)
unique["dec_round"] = np.round(unique["dec"].astype(float), 6)
_, first = np.unique(
    np.array([unique["ra_round"], unique["dec_round"]]).T,
    axis=0,
    return_index=True,
)
unique = unique[np.sort(first)]

print(
    f"{matched.sum()} / {len(dja)} DJA rows within {MATCH_RADIUS.to_value(u.arcsec):.0f}\" "
    f"({len(unique)} unique positions)"
)
out_cols = [
    "root", "srcid", "ra", "dec", "z", "grade", "sep_arcsec",
    "miri_obs_id", "miri_instrument_name", "miri_proposal_id", "miri_filters",
]
out_cols = [c for c in out_cols if c in unique.colnames]
unique[out_cols].write(DATA_DIR / "miri_dja_imag_matches_1arcmin.csv", overwrite=True)
print("wrote data/miri_dja_imag_matches_1arcmin.csv")
unique[out_cols]

126 / 177 DJA rows within 60" (57 unique positions)
wrote data/miri_dja_imag_matches_1arcmin.csv


root,srcid,ra,dec,z,grade,sep_arcsec,miri_obs_id,miri_instrument_name,miri_proposal_id,miri_filters
str24,int64,float64,float64,float64,int64,float64,str42,str10,str5,str7
uncover-v4,26185,3.5670708,-30.37786065,10.0628,3,0.0016632249475548048,jw06123-o005_t001_miri_f1800w,MIRI/IMAGE,6123,F1800W
uncover-v4,37126,3.59011077,-30.35974219,10.3781,3,0.0018942440545210869,jw09165-o004_t001_miri_f560w,MIRI/IMAGE,9165,F560W
ceers-ddt-v4,10,214.906633,52.945504,11.6351,3,0.022550526798963806,jw04586-o003_t001_miri_f770w,MIRI/IMAGE,4586,F770W
abell2744-castellano1-v4,22600,3.4989827,-30.3247534,12.3393,3,0.027945093548417507,jw07078-o002_t001_miri_f1000w,MIRI/IMAGE,7078,F1000W
capers-egs61-v4,22637,214.9320749,52.8418712,10.5927,3,0.03301011547302036,jw09165-o014_t005_miri_f560w,MIRI/IMAGE,9165,F560W
uncover-v4,3686,3.617202,-30.42553381,9.319,3,0.03312953388943966,jw04530-o002_t001_miri_f560w,MIRI/IMAGE,4530,F560W
goodsn-wide0-v4,1268,189.10605425,62.24204894,10.6056,3,0.054267873199693444,jw09165-o036_t013_miri_f560w,MIRI/IMAGE,9165,F560W
macs1149-stiavelli-v4,3,177.38990784,22.41271019,9.1112,3,0.06384361358935883,jw04530-o004_t002_miri_f770w,MIRI/IMAGE,4530,F770W
jades-gdn11-v4,5591,189.1060426,62.2420448,10.5915,3,0.0785082391128076,jw09165-o036_t013_miri_f560w,MIRI/IMAGE,9165,F560W


## 4. Download MIRI Level-3 `i2d` FITS

For each unique match, pull MAST science **`I2D`** products at `calib_level=3` (drizzled imaging).
Prefer broad-band filters in `PREFERRED_FILTERS` when several exist.

Some MAST rows are visit placeholders (`*_xx*` in `obs_id`) with no products; for those we fall back to the nearest association-level MIRI/IMAGE pointing that has an L3 `i2d`.

Programs still under exclusive access return HTTP 401; those are logged as `proprietary`.

In [5]:
def find_l3_i2d_products(obs_id):
    """Return MAST product table for L3 SCIENCE I2D FITS for one obs_id (or empty)."""
    obs_rows = Observations.query_criteria(obs_collection="JWST", obs_id=obs_id)
    if len(obs_rows) == 0:
        return obs_rows
    products = Observations.get_product_list(obs_rows)
    return Observations.filter_products(
        products,
        productType="SCIENCE",
        productSubGroupDescription="I2D",
        calib_level=[3],
        extension="fits",
    )


def resolve_download_obs_id(matched_obs_id, ra_deg, dec_deg, search_radius="2m"):
    """Prefer the matched obs_id; if it has no L3 I2D, use nearest association with one."""
    products = find_l3_i2d_products(matched_obs_id)
    if len(products) > 0:
        return matched_obs_id, products, "matched"

    # Placeholder / incomplete MAST rows: search nearby MIRI imaging
    nearby = Observations.query_criteria(
        obs_collection="JWST",
        instrument_name="MIRI/IMAGE",
        coordinates=f"{ra_deg} {dec_deg}",
        radius=search_radius,
    )
    if len(nearby) == 0:
        return matched_obs_id, products, "missing"

    # Association products look like jwPPPPP-oOOO_tTTT_miri_...
    is_assoc = np.array([("-o" in oid) and ("xx" not in oid) for oid in nearby["obs_id"]])
    candidates = nearby[is_assoc] if is_assoc.any() else nearby
    target = SkyCoord(ra_deg, dec_deg, unit="deg")
    cand_coords = SkyCoord(candidates["s_ra"], candidates["s_dec"], unit="deg")
    order = np.argsort(target.separation(cand_coords))

    for index in order:
        candidate_obs_id = candidates["obs_id"][index]
        candidate_products = find_l3_i2d_products(candidate_obs_id)
        if len(candidate_products) > 0:
            return candidate_obs_id, candidate_products, "fallback"

    return matched_obs_id, products, "missing"


def pick_i2d_products(products):
    """Prefer preferred-filter I2Ds; otherwise keep all L3 I2D rows."""
    if len(products) == 0:
        return products
    # Filter name often lives in productFilename (..._f770w_i2d.fits) or filters column
    names = [str(name).upper() for name in products["productFilename"]]
    preferred_mask = np.array([
        any(filt.lower() in name.lower() for filt in PREFERRED_FILTERS)
        for name in names
    ])
    if preferred_mask.any():
        return products[preferred_mask]
    return products

In [6]:
matches = Table.read(DATA_DIR / "miri_dja_imag_matches_1arcmin.csv")
FITS_DIR.mkdir(parents=True, exist_ok=True)

# One resolve/download pass per unique matched MIRI obs_id
unique_obs = matches[np.unique(matches["miri_obs_id"], return_index=True)[1]]
download_rows = []

for row in unique_obs:
    matched_obs_id = str(row["miri_obs_id"])
    download_obs_id, products, status = resolve_download_obs_id(
        matched_obs_id, float(row["ra"]), float(row["dec"])
    )
    chosen = pick_i2d_products(products) if status != "missing" else products

    if status == "missing" or len(chosen) == 0:
        print(f"{matched_obs_id}: {status} -> NONE")
        download_rows.append({
            "miri_obs_id_matched": matched_obs_id,
            "miri_obs_id_download": download_obs_id,
            "resolve_status": status if status != "matched" else "missing",
            "productFilename": "",
            "filter": "",
            "local_fits": "",
        })
        continue

    # Download every chosen I2D (often one file per filter)
    for product in chosen:
        product_name = str(product["productFilename"])
        # Infer filter from filename (..._f770w_i2d.fits)
        filt = ""
        lower = product_name.lower()
        for candidate in sorted(PREFERRED_FILTERS):
            if candidate.lower() in lower:
                filt = candidate
                break
        chosen_table = chosen[chosen["productFilename"] == product_name]
        manifest = Observations.download_products(
            chosen_table,
            download_dir=str(FITS_DIR),
            mrp_only=False,
        )
        local_path = str(manifest["Local Path"][0])
        download_status = str(manifest["Status"][0])
        message = str(manifest["Message"][0]) if "Message" in manifest.colnames else ""
        row_status = status
        if download_status != "COMPLETE" or not Path(local_path).is_file():
            row_status = "proprietary" if "401" in message else "download_failed"
            local_path = ""
            print(f"{matched_obs_id}: {row_status} ({download_status}) {product_name} {message}")
        else:
            print(f"{matched_obs_id}: {row_status} -> {product_name} ({filt or '?'})")

        download_rows.append({
            "miri_obs_id_matched": matched_obs_id,
            "miri_obs_id_download": download_obs_id,
            "resolve_status": row_status,
            "productFilename": product_name,
            "filter": filt,
            "local_fits": local_path,
        })

download_log = Table(rows=download_rows)
download_log.write(DATA_DIR / "miri_i2d_download_log.csv", overwrite=True)
n_ok = sum(1 for record in download_rows if record["local_fits"])
print(f"downloaded {n_ok} / {len(download_rows)} I2D products")
download_log

jw01180-o219_t219_miri_f1500w: matched -> jw01180-o219_t219_miri_f1500w_i2d.fits (F1500W)
INFO: Found cached file data/miri_i2d/mastDownload/JWST/jw01837-o008_t010_miri_f1800w/jw01837-o008_t010_miri_f1800w_i2d.fits with expected size 30116160. [astroquery.query]
jw01837-o008_t010_miri_f1800w: matched -> jw01837-o008_t010_miri_f1800w_i2d.fits (F1800W)
INFO: Found cached file data/miri_i2d/mastDownload/JWST/jw01837-o009_t011_miri_f770w/jw01837-o009_t011_miri_f770w_i2d.fits with expected size 30139200. [astroquery.query]
jw01837-o009_t011_miri_f770w: matched -> jw01837-o009_t011_miri_f770w_i2d.fits (F770W)
jw03794-o005_t004_miri_f1000w: matched -> jw03794-o005_t004_miri_f1000w_i2d.fits (F1000W)
INFO: Found cached file data/miri_i2d/mastDownload/JWST/jw03794-o006_t006_miri_f1500w/jw03794-o006_t006_miri_f1500w_i2d.fits with expected size 35798400. [astroquery.query]
jw03794-o006_t006_miri_f1500w: matched -> jw03794-o006_t006_miri_f1500w_i2d.fits (F1500W)
INFO: Found cached file data/miri_i2

miri_obs_id_matched,miri_obs_id_download,resolve_status,productFilename,filter,local_fits
str29,str29,str11,str38,str6,str100
jw01180-o219_t219_miri_f1500w,jw01180-o219_t219_miri_f1500w,matched,jw01180-o219_t219_miri_f1500w_i2d.fits,F1500W,data/miri_i2d/mastDownload/JWST/jw01180-o219_t219_miri_f1500w/jw01180-o219_t219_miri_f1500w_i2d.fits
jw01837-o008_t010_miri_f1800w,jw01837-o008_t010_miri_f1800w,matched,jw01837-o008_t010_miri_f1800w_i2d.fits,F1800W,data/miri_i2d/mastDownload/JWST/jw01837-o008_t010_miri_f1800w/jw01837-o008_t010_miri_f1800w_i2d.fits
jw01837-o009_t011_miri_f770w,jw01837-o009_t011_miri_f770w,matched,jw01837-o009_t011_miri_f770w_i2d.fits,F770W,data/miri_i2d/mastDownload/JWST/jw01837-o009_t011_miri_f770w/jw01837-o009_t011_miri_f770w_i2d.fits
jw03794-o005_t004_miri_f1000w,jw03794-o005_t004_miri_f1000w,matched,jw03794-o005_t004_miri_f1000w_i2d.fits,F1000W,data/miri_i2d/mastDownload/JWST/jw03794-o005_t004_miri_f1000w/jw03794-o005_t004_miri_f1000w_i2d.fits
jw03794-o006_t006_miri_f1500w,jw03794-o006_t006_miri_f1500w,matched,jw03794-o006_t006_miri_f1500w_i2d.fits,F1500W,data/miri_i2d/mastDownload/JWST/jw03794-o006_t006_miri_f1500w/jw03794-o006_t006_miri_f1500w_i2d.fits
jw03794-o018_t018_miri_f1000w,jw03794-o018_t018_miri_f1000w,matched,jw03794-o018_t018_miri_f1000w_i2d.fits,F1000W,data/miri_i2d/mastDownload/JWST/jw03794-o018_t018_miri_f1000w/jw03794-o018_t018_miri_f1000w_i2d.fits
jw03794-o023_t023_miri_f1500w,jw03794-o023_t023_miri_f1500w,matched,jw03794-o023_t023_miri_f1500w_i2d.fits,F1500W,data/miri_i2d/mastDownload/JWST/jw03794-o023_t023_miri_f1500w/jw03794-o023_t023_miri_f1500w_i2d.fits
jw04498-o106_t003_miri_f560w,jw04498-o106_t003_miri_f560w,matched,jw04498-o106_t003_miri_f560w_i2d.fits,F560W,data/miri_i2d/mastDownload/JWST/jw04498-o106_t003_miri_f560w/jw04498-o106_t003_miri_f560w_i2d.fits
jw04530-o002_t001_miri_f560w,jw04530-o002_t001_miri_f560w,matched,jw04530-o002_t001_miri_f560w_i2d.fits,F560W,data/miri_i2d/mastDownload/JWST/jw04530-o002_t001_miri_f560w/jw04530-o002_t001_miri_f560w_i2d.fits


## 5. Ready catalog for morphology / cutouts

Join DJA matches to successful downloads. One row per (source × filter) with a local `i2d` path.

In [7]:
matches = Table.read(DATA_DIR / "miri_dja_imag_matches_1arcmin.csv")
download_log = Table.read(DATA_DIR / "miri_i2d_download_log.csv")

# Successful downloads keyed by matched obs_id (may be several filters)
ok_mask = np.array([
    bool(str(path)) and Path(str(path)).is_file()
    for path in download_log["local_fits"]
])
ok_log = download_log[ok_mask]

prepared_rows = []
for source in matches:
    matched_obs_id = str(source["miri_obs_id"])
    source_tag = f"{source['root']}_{source['srcid']}"
    products = ok_log[ok_log["miri_obs_id_matched"] == matched_obs_id]

    if len(products) == 0:
        # Preserve a row even when download failed / proprietary
        log_hits = download_log[download_log["miri_obs_id_matched"] == matched_obs_id]
        status = str(log_hits["resolve_status"][0]) if len(log_hits) else "no_fits"
        prepared_rows.append({
            "root": source["root"],
            "srcid": int(source["srcid"]),
            "ra": float(source["ra"]),
            "dec": float(source["dec"]),
            "z": float(source["z"]),
            "grade": int(source["grade"]),
            "sep_arcsec": float(source["sep_arcsec"]),
            "miri_obs_id_matched": matched_obs_id,
            "miri_obs_id_download": "",
            "miri_instrument_name": str(source["miri_instrument_name"]),
            "filter": "",
            "local_fits": "",
            "status": status,
        })
        print(f"{source_tag}: {status}")
        continue

    for product in products:
        prepared_rows.append({
            "root": source["root"],
            "srcid": int(source["srcid"]),
            "ra": float(source["ra"]),
            "dec": float(source["dec"]),
            "z": float(source["z"]),
            "grade": int(source["grade"]),
            "sep_arcsec": float(source["sep_arcsec"]),
            "miri_obs_id_matched": matched_obs_id,
            "miri_obs_id_download": str(product["miri_obs_id_download"]),
            "miri_instrument_name": str(source["miri_instrument_name"]),
            "filter": str(product["filter"]),
            "local_fits": str(product["local_fits"]),
            "status": "ok",
        })
        print(f"{source_tag}: ok {product['filter']} -> {product['productFilename']}")

prepared = Table(rows=prepared_rows)
prepared.write(DATA_DIR / "miri_dja_imag_ready.csv", overwrite=True)
n_ok = sum(prepared["status"] == "ok")
print(f"ready {n_ok} / {len(prepared)} rows -> {DATA_DIR / 'miri_dja_imag_ready.csv'}")
prepared["root", "srcid", "z", "filter", "miri_obs_id_download", "status"]

uncover-v4_26185: ok F1800W -> jw06123-o005_t001_miri_f1800w_i2d.fits
uncover-v4_37126: proprietary
ceers-ddt-v4_10: ok F770W -> jw04586-o003_t001_miri_f770w_i2d.fits
abell2744-castellano1-v4_22600: proprietary
capers-egs61-v4_22637: proprietary
uncover-v4_3686: ok F560W -> jw04530-o002_t001_miri_f560w_i2d.fits
goodsn-wide0-v4_1268: proprietary
macs1149-stiavelli-v4_3: ok F770W -> jw04530-o004_t002_miri_f770w_i2d.fits
jades-gdn11-v4_5591: proprietary
macs1149-v4_5110240: ok F770W -> jw04530-o004_t002_miri_f770w_i2d.fits
jades-gdn10-v4_619: ok F2100W -> jw05407-o019_t064_miri_f2100w_i2d.fits
gds-udeep-v4_20128771: ok F1500W -> jw05279-o003_t006_miri_f1500w_i2d.fits
rubies-egs52-v4_952693: ok F1500W -> jw07814-o026_t025_miri_f1500w_i2d.fits
jades-gds03-v4_20083087: ok F1000W -> jw05407-o034_t079_miri_f1000w_i2d.fits
gds-udeep-v4_265801: ok F1000W -> jw05407-o039_t084_miri_f1000w_i2d.fits
gds-deep-v4_58975: ok F1000W -> jw05407-o039_t084_miri_f1000w_i2d.fits
gds-udeep-v4_20096216: ok F150

root,srcid,z,filter,miri_obs_id_download,status
str24,int64,float64,str6,str29,str11
uncover-v4,26185,10.0628,F1800W,jw06123-o005_t001_miri_f1800w,ok
uncover-v4,37126,10.3781,,,proprietary
ceers-ddt-v4,10,11.6351,F770W,jw04586-o003_t001_miri_f770w,ok
abell2744-castellano1-v4,22600,12.3393,,,proprietary
capers-egs61-v4,22637,10.5927,,,proprietary
uncover-v4,3686,9.319,F560W,jw04530-o002_t001_miri_f560w,ok
goodsn-wide0-v4,1268,10.6056,,,proprietary
macs1149-stiavelli-v4,3,9.1112,F770W,jw04530-o004_t002_miri_f770w,ok
jades-gdn11-v4,5591,10.5915,,,proprietary


## 6. Matching SEDs + wavelength overlap

Join imaging (`miri_dja_imag_ready`) to spectroscopy (`miri_dja_spectra_ready`) on `(root, srcid)`, then check whether each imaging **filter bandpass** overlaps the spectrum's wavelength range.

Overlap = filter interval ∩ spectrum `[λ_min, λ_max]`. Also report whether any spectrum points fall inside the filter window (usable for SED weighting in that band).

In [9]:
# Approximate MIRI imager bandpasses [um] (half-power-ish edges).
FILTER_WAVE_UM = {
    "F560W": (4.9, 6.4),
    "F770W": (6.4, 8.8),
    "F1000W": (9.0, 11.0),
    "F1130W": (10.9, 11.7),
    "F1280W": (11.7, 13.9),
    "F1500W": (13.5, 16.6),
    "F1800W": (16.0, 19.8),
    "F2100W": (18.5, 23.5),
    "F2550W": (23.0, 27.5),
}


def spectrum_wave_range(csv_path):
    """Return (wave_min, wave_max, n_pix) from an exported x1d CSV, or None."""
    path = Path(csv_path)
    if not path.is_file():
        return None
    raw = pd.read_csv(path)
    cols = {c.strip().lower(): c for c in raw.columns}
    if "wavelength" not in cols:
        return None
    wave = raw[cols["wavelength"]].to_numpy(float)
    wave = wave[np.isfinite(wave) & (wave > 0)]
    if len(wave) == 0:
        return None
    return float(wave.min()), float(wave.max()), int(len(wave))


def intervals_overlap(a_lo, a_hi, b_lo, b_hi):
    lo = max(a_lo, b_lo)
    hi = min(a_hi, b_hi)
    return lo < hi, float(max(hi - lo, 0.0))


imag = pd.read_csv(DATA_DIR / "miri_dja_imag_ready.csv")
spec = pd.read_csv(DATA_DIR / "miri_dja_spectra_ready.csv")
imag = imag[imag["status"].astype(str).str.lower() == "ok"].copy()
spec = spec[spec["status"].astype(str).str.lower() == "ok"].copy()

hit = imag.merge(
    spec[["root", "srcid", "spectrum_csv", "miri_instrument_name"]],
    on=["root", "srcid"],
    how="inner",
    suffixes=("", "_spec"),
)

rows = []
for _, row in hit.iterrows():
    filt = str(row["filter"]).upper()
    filt_lo, filt_hi = FILTER_WAVE_UM.get(filt, (np.nan, np.nan))
    spec_info = spectrum_wave_range(row["spectrum_csv"])
    if spec_info is None:
        spec_lo = spec_hi = np.nan
        n_pix = 0
        overlap = False
        overlap_um = 0.0
        n_in_filter = 0
        note = "missing_or_empty_spectrum_csv"
    else:
        spec_lo, spec_hi, n_pix = spec_info
        note = ""
        if not np.isfinite(filt_lo):
            overlap, overlap_um = False, 0.0
            n_in_filter = 0
            note = f"unknown_filter:{filt}"
        else:
            overlap, overlap_um = intervals_overlap(filt_lo, filt_hi, spec_lo, spec_hi)
            # Count spectrum samples inside the filter window.
            raw = pd.read_csv(row["spectrum_csv"])
            cols = {c.strip().lower(): c for c in raw.columns}
            wave = raw[cols["wavelength"]].to_numpy(float)
            n_in_filter = int(np.sum(np.isfinite(wave) & (wave >= filt_lo) & (wave <= filt_hi)))

    rows.append({
        "root": row["root"],
        "srcid": row["srcid"],
        "z": row["z"],
        "filter": filt,
        "filt_lo_um": filt_lo,
        "filt_hi_um": filt_hi,
        "spec_lo_um": spec_lo,
        "spec_hi_um": spec_hi,
        "n_spec_pix": n_pix,
        "n_in_filter": n_in_filter,
        "overlap": overlap,
        "overlap_um": overlap_um,
        "spectrum_csv": row["spectrum_csv"],
        "spec_instrument": row.get("miri_instrument_name", ""),
        "note": note,
    })

overlap_df = pd.DataFrame(rows)
out_csv = DATA_DIR / "miri_imag_spec_wavelength_overlap.csv"
overlap_df.to_csv(out_csv, index=False)

n_hit = len(overlap_df)
n_overlap = int(overlap_df["overlap"].sum()) if n_hit else 0
print(f"imaging–spectroscopy matches: {n_hit}")
print(f"with filter–spectrum wavelength overlap: {n_overlap} / {n_hit}")
print(f"wrote {out_csv}")
display_cols = [
    "root", "srcid", "filter", "filt_lo_um", "filt_hi_um",
    "spec_lo_um", "spec_hi_um", "n_in_filter", "overlap", "overlap_um", "note",
]
overlap_df[display_cols].sort_values(["overlap", "root", "srcid"], ascending=[False, True, True])

imaging–spectroscopy matches: 10
with filter–spectrum wavelength overlap: 6 / 10
wrote data/miri_imag_spec_wavelength_overlap.csv


,root,srcid,filter,filt_lo_um,filt_hi_um,spec_lo_um,spec_hi_um,n_in_filter,overlap,overlap_um,note
9,ceers-ddt-v4,1,F1000W,9.0,11.0,3.335170,14.002575,85,True,2.000000,
1,ceers-ddt-v4,10,F770W,6.4,8.8,8.670650,10.130550,100,True,0.129350,
6,jades-gds-wide2-v4,68,F1500W,13.5,16.6,3.335025,14.001967,36,True,0.501967,
7,jades-gds-wide2-v4,596,F1500W,13.5,16.6,3.335025,14.001967,36,True,0.501967,
5,jades-gds-wide2-v4,749,F1500W,13.5,16.6,3.335025,14.001967,36,True,0.501967,
8,jades-gds1-v4,20088041,F1500W,13.5,16.6,3.335025,14.001967,36,True,0.501967,
3,macs1149-stiavelli-v4,3,F770W,6.4,8.8,15.411250,17.978750,0,False,0.000000,
4,macs1149-v4,5110240,F770W,6.4,8.8,15.411250,17.978750,0,False,0.000000,
2,uncover-v4,3686,F560W,4.9,6.4,24.403000,28.699000,0,False,0.000000,
0,uncover-v4,26185,F1800W,16.0,19.8,3.334931,14.001574,0,False,0.000000,
